# 💰 Customer Lifetime Value (CLV) — BG/NBD + Gamma-Gamma

**Phase 5** of the Customer Intelligence Platform.

This notebook uses two probabilistic models to predict how much each customer is worth over future horizons (3 / 6 / 12 months):

| Model | Predicts |
|-------|---------|
| **BG/NBD** (Beta-Geometric / Neg-Binomial) | Probability a customer is still "alive" and their expected future transaction count |
| **Gamma-Gamma** | Expected average monetary value per transaction (independent of frequency) |

**CLV = predicted transactions × expected avg. value**

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import preprocess
from src.ltv_model import build_summary_data, fit_ltv_models, predict_clv, save_models
from src.visualizations import set_plot_style, save_fig

set_plot_style()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("✅ Setup complete")

## 1. Load & clean transactions

In [ ]:
df = preprocess(keep_returns=False, save=False)
print(f"Cleaned transactions: {len(df):,} rows")

## 2. Build lifetimes summary

Convert to the ``lifetimes`` summary format: ``frequency``, ``recency``, ``T`` (age), ``monetary_value``.

In [ ]:
summary = build_summary_data(df)
print("\nRepeat customers (freq > 0):", (summary["frequency"] > 0).sum())
summary.describe()

## 3. Fit BG/NBD + Gamma-Gamma

In [ ]:
summary, bgf, ggf = fit_ltv_models(summary)
summary.head()

## 4. CLV distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ["clv_3m", "clv_6m", "clv_12m"]):
    sns.histplot(summary[col].clip(lower=0), bins=50, kde=True, ax=ax)
    ax.set_title(col.replace("_", " ").upper())
    ax.set_xlabel("Predicted CLV (£)")
fig.suptitle("Predicted CLV distributions by horizon", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "clv_distributions", subdir="ltv")
plt.show()

## 5. Top customers by 12-month CLV

In [ ]:
top_clv = summary.nlargest(15, "clv_12m")[["frequency", "recency", "monetary_value", "clv_3m", "clv_6m", "clv_12m"]]
top_clv

## 6. Alive probability (BG/NBD)

Which customers are still "alive" and likely to purchase again?

In [ ]:
summary["alive_prob"] = bgf.conditional_probability_alive(
    summary["frequency"], summary["recency"], summary["T"]
)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(summary["alive_prob"], bins=30, kde=True, ax=ax, color="#4C72B0")
ax.set_title("Customer alive probability (BG/NBD)")
ax.set_xlabel("P(alive)")
save_fig(fig, "alive_probability", subdir="ltv")
plt.show()

In [ ]:
print("At-risk customers (P(alive) < 0.3):", (summary["alive_prob"] < 0.3).sum())
print("Very alive customers (P(alive) > 0.9):", (summary["alive_prob"] > 0.9).sum())

## 7. Expected transactions over time

Visualise cumulative expected purchases for the top-5 CLV customers.

In [ ]:
top5_ids = summary.nlargest(5, "clv_12m").index
periods = np.arange(1, 365)

fig, ax = plt.subplots(figsize=(12, 5))
for cid in top5_ids:
    row = summary.loc[cid]
    pred = bgf.conditional_expected_number_of_purchases_up_to_time(
        periods, row["frequency"], row["recency"], row["T"]
    )
    ax.plot(periods, pred, label=str(cid))
ax.set_xlabel("Days into the future")
ax.set_ylabel("Expected purchases")
ax.set_title("Expected future transactions — top 5 CLV customers")
ax.legend(title="Customer")
save_fig(fig, "expected_transactions_top5", subdir="ltv")
plt.show()

## 8. Save models for dashboard reuse

Persist the fitted BG/NBD and Gamma-Gamma models so the Streamlit dashboard can reload them without retraining.

In [ ]:
save_models(bgf, ggf)

## 9. Summary & next steps

### Key takeaways
- **Probabilistic CLV** gives per-customer predictions with confidence, unlike simple regression.
- The **alive probability** from BG/NBD naturally feeds the churn model (Phase 6).
- **Top CLV customers** are strong candidates for loyalty programs; **low-alive** customers for re-engagement.
- Models are saved to ``models/*.pkl`` for the dashboard to consume.

### Next notebook
→ **`04_churn.ipynb`** (Phase 6): predict churn probability with XGBoost + SHAP explanations.

---
© 2026 AdamAI-Systems.